# 后效大盘分布分析 —— 消耗类

**数据来源**：`ad_callback_log_from_ad_log_full` 单日数据，预先用 Hive SQL 导出为 CSV  
**分析维度**：`visitor_id`  
**分箱方式**：等频分箱（Quantile），10个分箱  
**覆盖口径**：总消耗 / 正常流量消耗 / 作弊流量消耗 / 内循环消耗 / 外循环CID消耗 / 线索消耗 / 下载消耗

## Step 0：前置 Hive SQL（在 Hive 上执行，结果下载为 CSV）

```sql
SELECT
    visitor_id,
    llsid,
    creative_id,
    unit_id,
    action_type,
    charge_action_type,
    ocpc_action_type,
    is_for_report_engine,
    is_duplicate,
    is_retry,
    cost_total,
    callback_purchase_amount,
    clue_id,
    is_repeat_clue
FROM ad_callback_log_from_ad_log_full
WHERE p_date = '${p_date}'
  AND is_duplicate = false
  AND is_retry = false
```

> 下载后将文件命名为 `ad_callback_log.csv`，放在本 notebook 同目录下

## Step 1：配置区

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

CSV_PATH = "ad_callback_log.csv"
N_BINS = 10

INCYCLE_OCPC_TYPES = {
    'AD_MERCHANT_ROAS', 'EVENT_ORDER_PAIED',
    'AD_STOREWIDE_ROAS', 'AD_MERCHANT_T7_ROI', 'AD_FANS_TOP_ROI'
}

EXCYCLE_CID_OCPC_TYPES = {
    'EVENT_ORDER_SUBMIT', 'AD_CID_ROAS', 'CID_ROAS', 'CID_EVENT_ORDER_PAID'
}

CLUE_OCPC_TYPES = {
    'AD_LANDING_PAGE_FORM_SUBMITTED', 'EVENT_VALID_CLUES',
    'AD_EFFECTIVE_CUSTOMER_ACQUISITION', 'EVENT_INTENTION_CONFIRMED',
    'LEADS_SUBMIT', 'EVENT_PHONE_GET_THROUGH'
}

DOWNLOAD_OCPC_TYPES = {
    'AD_CONVERSION', 'AD_PURCHASE', 'AD_PURCHASE_CONVERSION',
    'AD_ROAS', 'AD_SEVEN_DAY_ROAS',
    'AD_ITEM_DOWNLOAD_COMPLETED', 'AD_ITEM_CLICK_DOWNLOAD'
}

## Step 2：工具函数

In [ ]:
def qcut_distribution(uv_df, value_col, visitor_col='visitor_id', n_bins=N_BINS):
    uv_df = uv_df[uv_df[value_col] > 0].copy()
    if uv_df.empty:
        print(f"  [WARN] {value_col}: 无有效数据")
        return pd.DataFrame()
    uv_df['bin'] = pd.qcut(uv_df[value_col], q=n_bins, duplicates='drop')
    result = (
        uv_df.groupby('bin', observed=True)
        .agg(
            cnt=(visitor_col, 'count'),
            min_val=(value_col, 'min'),
            max_val=(value_col, 'max'),
            avg_val=(value_col, 'mean'),
            total_val=(value_col, 'sum'),
        )
        .reset_index()
    )
    result['bin'] = result['bin'].astype(str)
    result['pct'] = (result['cnt'] / result['cnt'].sum() * 100).round(2)
    result.columns = ['分箱区间', '用户数cnt', '最小值', '最大值', '均值', '总计', '占比%']
    return result


def show_distribution(result, metric_name, save_csv=True):
    print(f"\n{'='*65}")
    print(f"  {metric_name}  （visitor_id维度，等频{N_BINS}箱）")
    print(f"{'='*65}")
    if result.empty:
        return
    print(result.to_string(index=False))
    if save_csv:
        fname = f"dist_{metric_name}.csv"
        result.to_csv(fname, index=False)
        print(f"  => 已保存: {fname}")

## Step 3：读取数据 & 基础预处理

In [ ]:
df = pd.read_csv(CSV_PATH)

# is_for_report_engine Hive导出可能是字符串，统一处理
if df['is_for_report_engine'].dtype == object:
    df['is_for_report_engine'] = df['is_for_report_engine'].str.lower() == 'true'

df['cost_yuan'] = df['cost_total'] / 1000.0

# 所有消耗口径的基础：计费行
df_charge = df[df['action_type'] == df['charge_action_type']].copy()

print(f"原始总行数:  {len(df):,}")
print(f"计费行数:    {len(df_charge):,}")
print(f"visitor_id去重数: {df_charge['visitor_id'].nunique():,}")

## Step 4：各消耗口径分布分析

### 4.1 总消耗（total_cost_yuan）

In [ ]:
uv = df_charge.groupby('visitor_id', as_index=False).agg(total_cost_yuan=('cost_yuan', 'sum'))
result_total = qcut_distribution(uv, 'total_cost_yuan')
show_distribution(result_total, 'total_cost_yuan')

### 4.2 正常流量消耗（normal_cost_yuan）
> `is_for_report_engine = true`

In [ ]:
uv = (
    df_charge[df_charge['is_for_report_engine'] == True]
    .groupby('visitor_id', as_index=False)
    .agg(normal_cost_yuan=('cost_yuan', 'sum'))
)
result_normal = qcut_distribution(uv, 'normal_cost_yuan')
show_distribution(result_normal, 'normal_cost_yuan')

### 4.3 作弊流量消耗（spam_cost_yuan）
> `is_for_report_engine = false`

In [ ]:
uv = (
    df_charge[df_charge['is_for_report_engine'] == False]
    .groupby('visitor_id', as_index=False)
    .agg(spam_cost_yuan=('cost_yuan', 'sum'))
)
result_spam = qcut_distribution(uv, 'spam_cost_yuan')
show_distribution(result_spam, 'spam_cost_yuan')

### 4.4 内循环消耗（incycle_cost_yuan）
> `ocpc_action_type IN ('AD_MERCHANT_ROAS','EVENT_ORDER_PAIED','AD_STOREWIDE_ROAS','AD_MERCHANT_T7_ROI','AD_FANS_TOP_ROI')`

In [ ]:
uv = (
    df_charge[df_charge['ocpc_action_type'].isin(INCYCLE_OCPC_TYPES)]
    .groupby('visitor_id', as_index=False)
    .agg(incycle_cost_yuan=('cost_yuan', 'sum'))
)
result_incycle = qcut_distribution(uv, 'incycle_cost_yuan')
show_distribution(result_incycle, 'incycle_cost_yuan')

### 4.5 外循环CID消耗（excycle_cid_cost_yuan）
> `ocpc_action_type IN ('EVENT_ORDER_SUBMIT','AD_CID_ROAS','CID_ROAS','CID_EVENT_ORDER_PAID')`

In [ ]:
uv = (
    df_charge[df_charge['ocpc_action_type'].isin(EXCYCLE_CID_OCPC_TYPES)]
    .groupby('visitor_id', as_index=False)
    .agg(excycle_cid_cost_yuan=('cost_yuan', 'sum'))
)
result_excycle = qcut_distribution(uv, 'excycle_cid_cost_yuan')
show_distribution(result_excycle, 'excycle_cid_cost_yuan')

### 4.6 线索消耗（clue_cost_yuan）
> `ocpc_action_type IN ('AD_LANDING_PAGE_FORM_SUBMITTED','EVENT_VALID_CLUES','AD_EFFECTIVE_CUSTOMER_ACQUISITION','EVENT_INTENTION_CONFIRMED','LEADS_SUBMIT','EVENT_PHONE_GET_THROUGH')`

In [ ]:
uv = (
    df_charge[df_charge['ocpc_action_type'].isin(CLUE_OCPC_TYPES)]
    .groupby('visitor_id', as_index=False)
    .agg(clue_cost_yuan=('cost_yuan', 'sum'))
)
result_clue = qcut_distribution(uv, 'clue_cost_yuan')
show_distribution(result_clue, 'clue_cost_yuan')

### 4.7 下载消耗（download_cost_yuan）
> `ocpc_action_type IN ('AD_CONVERSION','AD_PURCHASE','AD_PURCHASE_CONVERSION','AD_ROAS','AD_SEVEN_DAY_ROAS','AD_ITEM_DOWNLOAD_COMPLETED','AD_ITEM_CLICK_DOWNLOAD')`

In [ ]:
uv = (
    df_charge[df_charge['ocpc_action_type'].isin(DOWNLOAD_OCPC_TYPES)]
    .groupby('visitor_id', as_index=False)
    .agg(download_cost_yuan=('cost_yuan', 'sum'))
)
result_download = qcut_distribution(uv, 'download_cost_yuan')
show_distribution(result_download, 'download_cost_yuan')

## Step 5：汇总总览

In [ ]:
summary = []
for name, uv_df, col in [
    ('total_cost_yuan',       df_charge,                                                           'cost_yuan'),
    ('normal_cost_yuan',      df_charge[df_charge['is_for_report_engine'] == True],                'cost_yuan'),
    ('spam_cost_yuan',        df_charge[df_charge['is_for_report_engine'] == False],               'cost_yuan'),
    ('incycle_cost_yuan',     df_charge[df_charge['ocpc_action_type'].isin(INCYCLE_OCPC_TYPES)],   'cost_yuan'),
    ('excycle_cid_cost_yuan', df_charge[df_charge['ocpc_action_type'].isin(EXCYCLE_CID_OCPC_TYPES)],'cost_yuan'),
    ('clue_cost_yuan',        df_charge[df_charge['ocpc_action_type'].isin(CLUE_OCPC_TYPES)],      'cost_yuan'),
    ('download_cost_yuan',    df_charge[df_charge['ocpc_action_type'].isin(DOWNLOAD_OCPC_TYPES)],  'cost_yuan'),
]:
    agg = uv_df.groupby('visitor_id')[col].sum()
    agg = agg[agg > 0]
    summary.append({
        '口径': name,
        '有效UV数': len(agg),
        '总消耗(元)': round(agg.sum(), 2),
        '人均消耗(元)': round(agg.mean(), 4),
        'p50': round(agg.quantile(0.5), 4),
        'p90': round(agg.quantile(0.9), 4),
        'p99': round(agg.quantile(0.99), 4),
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv('cost_summary.csv', index=False)